# SSAE Symbolic Logic Training

Trains the sparse autoencoder on symbolic logic reasoning traces.

**Runtime:** T4 (~45 min total) or A100 (~15 min total)  
**Output:** `ssae_symbolic_p1.enc.pt` (~3 MB) — download this file back to your local machine.

### Steps
1. Config & setup
2. Mount Google Drive (data + checkpoints persist across sessions)
3. Clone repo + install deps
4. Generate synthetic ProntoQA problems + model traces *(skipped if Drive copy exists)*
5. Train SSAE Phase 1
6. Download the inference checkpoint

## 0 — Config
Set your GitHub repo URL and training parameters here.

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
GITHUB_REPO   = "https://github.com/djaxchi/CoT-checker.git"

# Data generation
N_PROBLEMS    = 14000   # ~50K steps at avg 3.5 steps/problem
GEN_BATCH     = 16      # LLM generation batch size

# Training
TRAIN_EPOCHS  = 1
TRAIN_BATCH   = 16      # per-device batch size
GRAD_ACCUM    = 4       # effective batch = 16 * 4 = 64
# ── End config ────────────────────────────────────────────────────────────────

import torch
gpu  = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU!)"
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f"GPU : {gpu}")
print(f"VRAM: {vram:.0f} GB")

# A100 (40 GB) can unfreeze backbones; T4 (16 GB) must keep them frozen
FREEZE_BACKBONES = vram < 35
print(f"Freeze backbones: {FREEZE_BACKBONES}  ({'T4/V100 — memory constrained' if FREEZE_BACKBONES else 'A100 — full training'})")

## 1 — Mount Google Drive

Data and checkpoints are stored in `MyDrive/CoT-checker/` so they survive runtime resets.  
Generation is skipped automatically if the traces file already exists on Drive.

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT   = "/content/drive/MyDrive/CoT-checker"
DRIVE_DATA   = f"{DRIVE_ROOT}/data"
DRIVE_CKPT   = f"{DRIVE_ROOT}/results/checkpoints"
DRIVE_PROBE  = f"{DRIVE_ROOT}/results/probe_data"

for d in [DRIVE_DATA, DRIVE_CKPT, DRIVE_PROBE]:
    os.makedirs(d, exist_ok=True)

TRACES_PATH   = f"{DRIVE_DATA}/prontoqa_traces.jsonl"
PROBLEMS_PATH = f"{DRIVE_DATA}/prontoqa_synthetic.jsonl"
CKPT_PATH     = f"{DRIVE_CKPT}/ssae_symbolic_p1.pt"
ENC_PATH      = f"{DRIVE_CKPT}/ssae_symbolic_p1.enc.pt"
PROBE_PATH    = f"{DRIVE_PROBE}/symbolic_ssae_p1.npz"

traces_exist = os.path.exists(TRACES_PATH)
print(f"Traces on Drive: {'YES — generation will be skipped' if traces_exist else 'NO — will generate'}")
print(f"Checkpoint on Drive: {'YES' if os.path.exists(CKPT_PATH) else 'NO'}")

## 2 — Clone repo + install dependencies

In [ ]:
import os

if not os.path.isdir("/content/CoT-checker"):
    !git clone {GITHUB_REPO} /content/CoT-checker
else:
    !git -C /content/CoT-checker pull

%cd /content/CoT-checker

!pip install -q transformers datasets tqdm

## 3 — Generate synthetic ProntoQA problems + model traces

Phase A: generate problems algorithmically (no LLM needed, instant).  
Phase B: run Qwen2.5-0.5B on each problem to get model-generated CoT traces.

**Skipped automatically if `prontoqa_traces.jsonl` already exists on Drive.**

In [ ]:
import os

if os.path.exists(TRACES_PATH):
    print(f"Traces already exist at {TRACES_PATH} — skipping generation.")
else:
    !python scripts/generate_symbolic_data.py \
        --n-problems {N_PROBLEMS} \
        --problems-out {PROBLEMS_PATH} \
        --traces-out   {TRACES_PATH} \
        --batch-size {GEN_BATCH} \
        --device cuda
    print("\nGeneration complete. Files saved to Drive.")

In [ ]:
# Sanity check: inspect a few traces
import json

total_steps  = sum(len(json.loads(l)["steps"]) for l in open(TRACES_PATH))
total_traces = sum(1 for _ in open(TRACES_PATH))
print(f"Total traces : {total_traces}")
print(f"Total steps  : {total_steps}")
print()
with open(TRACES_PATH) as f:
    for s in [json.loads(next(f)) for _ in range(3)]:
        print(f"Q: {s['question']}")
        print(f"Steps: {s['steps']}")
        print()

## 4 — Train SSAE Phase 1

Trains the sparse autoencoder to reconstruct step tokens from a sparse latent `h_c`.  
The Qwen2.5-0.5B backbone is frozen on T4; unfrozen on A100.

Checkpoint is saved to Drive — safe to interrupt and resume.

In [ ]:
freeze_flag = "--freeze-backbones" if FREEZE_BACKBONES else "--no-freeze-backbones"

!python scripts/train_ssae_symbolic.py \
    --traces  {TRACES_PATH} \
    --output  {CKPT_PATH} \
    --phase 1 \
    --epochs {TRAIN_EPOCHS} \
    --batch-size {TRAIN_BATCH} \
    --grad-accum {GRAD_ACCUM} \
    --device cuda \
    {freeze_flag}

In [ ]:
# Verify checkpoint files on Drive
import os
for f in os.listdir(DRIVE_CKPT):
    path = f"{DRIVE_CKPT}/{f}"
    size = os.path.getsize(path) / 1e6
    print(f"{f:<50}  {size:6.1f} MB")

## 5 — Download the inference checkpoint

The `.enc.pt` file (~3 MB) is all you need locally to run the probe pipeline.  
Place it at `results/checkpoints/ssae_symbolic_p1.enc.pt` on your machine.

In [ ]:
from google.colab import files

# Download the inference checkpoint (~3 MB)
files.download(ENC_PATH)

# Optional: also download the traces for local probe-data generation
# files.download(TRACES_PATH)

## 6 — (Optional) Generate probe data on Colab and download

If you want to skip re-running generation locally, encode the traces here and download the `.npz` (~few MB).

In [ ]:
# This uses the full checkpoint (not the slim .enc.pt) so it can re-use the
# backbone that's already in memory on Colab.

!python scripts/generate_probe_data_symbolic.py \
    --checkpoint {CKPT_PATH} \
    --data       {PROBLEMS_PATH} \
    --output     {PROBE_PATH} \
    --device cuda

files.download(PROBE_PATH)